In [0]:
%sql
-- CREATE SCHEMA IF NOT EXISTS dbcertification.easewithdata
-- COMMENT 'Schema for ease with data'
-- LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbcertification.easewithdata.orders_raw DEEP CLONE samples.tpch.orders
LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/orders_raw';

CREATE TABLE IF NOT EXISTS dbcertification.easewithdata.customer_raw DEEP CLONE samples.tpch.customer
LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/customer_raw';

In [0]:
%sql
select * from dbcertification.easewithdata.orders_raw limit 5;

In [0]:
%sql
select * from dbcertification.easewithdata.customer_raw limit 5;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

**Add some data incrementally in ORDERS**

In [0]:
%sql
INSERT INTO dbcertification.easewithdata.orders_raw
SELECT * FROM samples.tpch.orders
LIMIT 10000;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

In [0]:
%sql
DESCRIBE HISTORY dbcertification.easewithdata.orders_bronze;

#### DLT with AutoLoader

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing/")

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing/files")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing/autoloader/schemas")

#### SCD Type 1 and Type 2 in DLT - Changes to be done in cutomer_bronze table

In [0]:
%sql
ALTER TABLE dbcertification.easewithdata.customer_raw
ADD COLUMNS
(
    __src_action STRING, -- Possible values Insert, Delete, Truncate
    __src_insert_dt TIMESTAMP
);
-- Any changes in data at the source side will be a new record appended into the source table

In [0]:
%sql
-- update those columns added in table 
UPDATE dbcertification.easewithdata.customer_raw
SET __src_action = 'I', 
    __src_insert_dt = current_timestamp() - interval '3 days';

In [0]:
%sql
-- display the table
SELECT * FROM dbcertification.easewithdata.customer_raw;